# Limpieza de datos - Clustering de Usuarios

Este notebook realiza la limpieza de datos específica para el **proyecto de clustering de usuarios**. A diferencia del proyecto de regresión, aquí:

- ✅ **CustomerID nulos**: Se ELIMINAN (requisito obligatorio - sin cliente no hay cluster)
- ✅ **Description nulas**: Se ELIMINAN (sin descripción no hay info del producto)
- ✅ **Cancelaciones**: Se CONSERVAN (la tasa de devolución es un patrón de comportamiento)
- ✅ **Outliers**: Capping al p99 (balance entre mantener datos y reducir extremos)

## Anomalías identificadas en el EDA:
- Registros duplicados
- Transacciones sin identificación de usuario (CustomerID nulo)
- Cantidades y precios unitarios negativos o cero
- Outliers en cantidades y precios unitarios
- StockCodes no estándar (códigos operacionales)

## 0. Imports y Carga del Dataset

Preparamos el entorno de trabajo: importamos librerías, configuramos la estética visual, definimos las rutas del proyecto y cargamos el dataset original. 

A continuación creamos las columnas auxiliares necesarias para los pasos de limpieza:
- `Fecha`: fecha sin hora (normalizada)
- `Mes`: periodo mensual
- `DiaSemana`: nombre del día de la semana
- `EsCancelacion`: booleano si el InvoiceNo empieza con 'C'
- `TotalPrice`: cantidad × precio unitario

Inicializamos el diccionario de auditoría `stats_cleaning` que registrará paso a paso cuántos registros eliminamos y por qué.

In [1]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns

# Configuración visual global
sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.figsize'] = (12, 5)
%matplotlib inline

# Rutas del proyecto
RUTA_CSV      = '../../../data/raw/data.csv'
RUTA_GRAFICOS = '../../../graphics/clean/'
RUTA_INTERIM  = '../../../data/interim/interim_ProyClustering/'
os.makedirs(RUTA_GRAFICOS, exist_ok=True)
os.makedirs(RUTA_INTERIM,  exist_ok=True)

print('Librerías cargadas correctamente.')
print(f'Rutas configuradas:')
print(f'  - CSV raw    : {RUTA_CSV}')
print(f'  - Gráficos   : {RUTA_GRAFICOS}')
print(f'  - Interim    : {RUTA_INTERIM}')

Librerías cargadas correctamente.
Rutas configuradas:
  - CSV raw    : ../../../data/raw/data.csv
  - Gráficos   : ../../../graphics/clean/
  - Interim    : ../../../data/interim/interim_ProyClustering/


In [2]:
# Carga del dataset original
df_raw     = pd.read_csv(RUTA_CSV, encoding='latin-1')
df_working = df_raw.copy()

# ── Columnas auxiliares ────────────────────────────────────────────────────────
df_working['InvoiceDate']   = pd.to_datetime(df_working['InvoiceDate'], format='mixed')
df_working['Fecha']         = df_working['InvoiceDate'].dt.normalize()
df_working['Mes']           = df_working['InvoiceDate'].dt.to_period('M')
df_working['DiaSemana']     = df_working['InvoiceDate'].dt.day_name()
df_working['EsCancelacion'] = df_working['InvoiceNo'].str.startswith('C')
df_working['TotalPrice']    = df_working['Quantity'] * df_working['UnitPrice']

# ── Diccionario de auditoría (se actualiza en cada paso) ──────────────────────
stats_cleaning = {'Registros Iniciales': len(df_raw)}

# ── Resumen de carga ──────────────────────────────────────────────────────────
print('=' * 55)
print(f'  DATASET CARGADO - CLUSTERING DE USUARIOS')
print('=' * 55)
print(f'  Filas    : {df_raw.shape[0]:>10,}')
print(f'  Columnas : {df_raw.shape[1]:>10}')
print('=' * 55)
print(f'\n  df_working activo : {len(df_working):,} filas')
print(f'  Columnas auxiliares añadidas:')
print(f'    - Fecha, Mes, DiaSemana, EsCancelacion, TotalPrice')
print(f'\n  Columnas originales: {list(df_raw.columns)}')

  DATASET CARGADO - CLUSTERING DE USUARIOS
  Filas    :    541,909
  Columnas :          8

  df_working activo : 541,909 filas
  Columnas auxiliares añadidas:
    - Fecha, Mes, DiaSemana, EsCancelacion, TotalPrice

  Columnas originales: ['InvoiceNo', 'StockCode', 'Description', 'Quantity', 'InvoiceDate', 'UnitPrice', 'CustomerID', 'Country']


In [3]:
# ── Inspección inicial de valores nulos ───────────────────────────────────────
print('\n' + '=' * 55)
print('  VALORES NULOS INICIALES')
print('=' * 55)

nulos_count = df_working.isnull().sum()
nulos_pct = (df_working.isnull().sum() / len(df_working) * 100).round(2)
nulos_df = pd.DataFrame({
    'Nulos': nulos_count,
    'Porcentaje': nulos_pct
})
nulos_df = nulos_df[nulos_df['Nulos'] > 0].sort_values('Nulos', ascending=False)

if len(nulos_df) > 0:
    print(f'\n  Columnas con valores nulos:')
    for col, row in nulos_df.iterrows():
        print(f"    {col:<15} : {row['Nulos']:>10,} ({row['Porcentaje']:>6.2f}%)")
else:
    print(f'\n  ✓ No hay valores nulos en el dataset')

print(f'\n  ⚠ CRÍTICO para clustering: CustomerID tiene {nulos_df.loc["CustomerID", "Nulos"] if "CustomerID" in nulos_df.index else 0:,} nulos → DEBEN ser eliminados')


  VALORES NULOS INICIALES

  Columnas con valores nulos:
    CustomerID      :  135,080.0 ( 24.93%)
    Description     :    1,454.0 (  0.27%)

  ⚠ CRÍTICO para clustering: CustomerID tiene 135,080 nulos → DEBEN ser eliminados


---
# 3. LIMPIEZA DE DATOS

A continuación se ejecutan los pasos de limpieza secuencialmente, actualizando el diccionario `stats_cleaning` en cada paso para mantener la auditoría completa del proceso.

### 3.1 Eliminar duplicados exactos

Una fila duplicada exacta tiene **todos** sus campos idénticos: mismo `InvoiceNo`, `StockCode`, `Description`, `Quantity`, `UnitPrice`, `InvoiceDate`, `CustomerID` y `Country`. 

**Justificación**: Esto es físicamente imposible en un sistema transaccional real — su presencia indica errores de doble inserción en la base de datos, exports corruptos o fallos de ETL. 

Se conserva la primera ocurrencia (`keep='first'`) al ser todas idénticas.

In [4]:
# 3.1 — Eliminar duplicados exactos
antes = len(df_working)
df_working = df_working.drop_duplicates(keep='first', ignore_index=True)
eliminadas_dup = antes - len(df_working)
stats_cleaning['Duplicados eliminados'] = eliminadas_dup

print(f"{'='*60}")
print(f"  3.1 — ELIMINAR DUPLICADOS EXACTOS")
print(f"{'='*60}")
print(f"  Justificación: Errores de inserción en BD/ETL")
print(f"  Estrategia   : Conservar primera ocurrencia (keep='first')")
print()
print(f"  Filas antes     : {antes:>10,}")
print(f"  Filas eliminadas: {eliminadas_dup:>10,}")
print(f"  Filas después   : {len(df_working):>10,}")
print(f"  Verificación    : duplicados restantes = {df_working.duplicated().sum()}")
print(f"{'='*60}")

  3.1 — ELIMINAR DUPLICADOS EXACTOS
  Justificación: Errores de inserción en BD/ETL
  Estrategia   : Conservar primera ocurrencia (keep='first')

  Filas antes     :    541,909
  Filas eliminadas:      5,268
  Filas después   :    536,641
  Verificación    : duplicados restantes = 0


### 3.2 Eliminar negativos huérfanos

Son filas con `Quantity < 0` pero **sin prefijo `C`** en `InvoiceNo`. En este dataset son ajustes internos de almacén (descripciones como "faulty", "damages", "check", "reverse adjustment"). 

**Justificación**: El análisis del CSV confirma que el **100%** cumple simultáneamente:
- `UnitPrice = 0.0` → `TotalPrice = 0` siempre, sin impacto en ventas
- `CustomerID = NaN` → ninguna tiene cliente asociado
- Sin prefijo `C` → el sistema nunca las registró como cancelación oficial

**Decisión**: No son cancelaciones de venta ni transacciones recuperables. No tienen cliente identificado → no aportan valor al clustering. Se eliminan.

In [5]:
# 3.2 — Eliminar negativos huérfanos (ajustes internos de almacén)
antes = len(df_working)
mask_huerfanos = (
    ~df_working['InvoiceNo'].str.startswith('C', na=False) &
    (df_working['Quantity'] < 0) &
    (df_working['UnitPrice'] == 0.0)
)
df_working = df_working[~mask_huerfanos].reset_index(drop=True)
eliminadas_huerfanos = antes - len(df_working)
stats_cleaning['Negativos huérfanos eliminados'] = eliminadas_huerfanos

print(f"{'='*60}")
print(f"  3.2 — ELIMINAR NEGATIVOS HUÉRFANOS")
print(f"{'='*60}")
print(f"  Justificación: Ajustes internos sin valor")
print(f"               : sin C + Qty<0 + Price=0 → sin cliente")
print(f"  Criterio     : ~InvoiceNo.startswith('C') & Quantity<0 & UnitPrice==0")
print()
print(f"  Filas antes     : {antes:>10,}")
print(f"  Filas eliminadas: {eliminadas_huerfanos:>10,}")
print(f"  Filas después   : {len(df_working):>10,}")
print(f"  Verificación    : huérfanos restantes = {(~df_working['InvoiceNo'].str.startswith('C', na=False) & (df_working['Quantity'] < 0) & (df_working['UnitPrice'] == 0.0)).sum()}")
print(f"{'='*60}")

  3.2 — ELIMINAR NEGATIVOS HUÉRFANOS
  Justificación: Ajustes internos sin valor
               : sin C + Qty<0 + Price=0 → sin cliente
  Criterio     : ~InvoiceNo.startswith('C') & Quantity<0 & UnitPrice==0

  Filas antes     :    536,641
  Filas eliminadas:      1,336
  Filas después   :    535,305
  Verificación    : huérfanos restantes = 0


### 3.3 Eliminar StockCodes no estándar

Los códigos de producto comerciales siguen el patrón `DDDDD[LL]` (5 dígitos + hasta 2 letras opcionales, ej: `22423`, `85123A`, `15056BL`).

**Justificación**: Códigos como `POST`, `DOT`, `D`, `M`, `BANK CHARGES`, `AMAZONFEE`, `C2` son códigos operacionales (portes, descuentos, comisiones, ajustes manuales) que **no representan ventas de productos reales**.

**Decisión**: Para clustering de usuarios por comportamiento de compra, necesitamos solo productos físicos reales. Los códigos operacionales no aportan patrones de consumo válidos. Se eliminan.

In [6]:
# 3.3 — Eliminar StockCodes no estándar
# Patrón: 5 dígitos + hasta 2 letras opcionales (cubre variantes como 15056BL)
PATRON_STOCK = r'^[0-9]{5}[A-Za-z]{0,2}$'

antes = len(df_working)
mask_estandar = df_working['StockCode'].str.match(PATRON_STOCK, na=False)
df_working = df_working[mask_estandar].reset_index(drop=True)
eliminadas_stock = antes - len(df_working)
stats_cleaning['StockCodes no estándar eliminados'] = eliminadas_stock

print(f"{'='*60}")
print(f"  3.3 — ELIMINAR STOCKCODES NO ESTÁNDAR")
print(f"{'='*60}")
print(f"  Justificación: Códigos operacionales no son productos reales")
print(f"               : Ejemplos: POST, DOT, BANK CHARGES, AMAZONFEE")
print(f"  Patrón válido: {PATRON_STOCK}")
print(f"               : 5 dígitos + hasta 2 letras opcionales")
print()
print(f"  Filas antes     : {antes:>10,}")
print(f"  Filas eliminadas: {eliminadas_stock:>10,}")
print(f"  Filas después   : {len(df_working):>10,}")
print(f"  Verificación    : no estándar restantes = {(~df_working['StockCode'].str.match(PATRON_STOCK, na=False)).sum()}")
print(f"{'='*60}")

  3.3 — ELIMINAR STOCKCODES NO ESTÁNDAR
  Justificación: Códigos operacionales no son productos reales
               : Ejemplos: POST, DOT, BANK CHARGES, AMAZONFEE
  Patrón válido: ^[0-9]{5}[A-Za-z]{0,2}$
               : 5 dígitos + hasta 2 letras opcionales

  Filas antes     :    535,305
  Filas eliminadas:      2,977
  Filas después   :    532,328
  Verificación    : no estándar restantes = 0
